## Step 0 — Gradient Checker

In [17]:
from __future__ import annotations  # lazy annotations: allows tuple[...] / X | None on older kernels

from typing import Callable
import numpy as np

In [37]:
def numeric_gradient(
    scalar_function: Callable[[np.ndarray], float],
    feature_values: np.ndarray,
    h: float = 1e-5
) -> np.ndarray:
    gradient = np.zeros_like(feature_values, dtype=float)

    for i in range(feature_values.size):
        mask = np.zeros_like(feature_values, dtype=float)
        mask.flat[i] = h
        gradient.flat[i] = (scalar_function(feature_values + mask) - scalar_function(feature_values - mask)) / (2 * h)
    return gradient

def stable_softmax(x: np.ndarray) -> np.ndarray:
    e = np.exp(x - np.max(x))
    return e / np.sum(e)

### Tests — `numeric_gradient`

In [38]:
def _check(name, got, want, atol=1e-6):
    ok = np.allclose(got, want, atol=atol)
    print(f"[{'PASS' if ok else 'FAIL'}] {name}")
    if not ok:
        print("   got :", got)
        print("   want:", want)

# 1. f = sum(x^2)  ->  grad = 2x        (vector input, nonlinear)
x = np.array([3.0, -1.0, 0.5])
_check("sum(x^2) grad == 2x", numeric_gradient(lambda v: np.sum(v**2), x), 2 * x)

# 2. f = sum(c*x)  ->  grad = c         (linear -> constant gradient)
c = np.array([2.0, -3.0, 0.7])
_check("sum(c*x) grad == c", numeric_gradient(lambda v: np.sum(c * v), np.zeros(3)), c)

# 3. matrix input -> gradient keeps the matrix shape
W = np.arange(6, dtype=float).reshape(2, 3)
g = numeric_gradient(lambda M: np.sum(M**2), W)
_check("matrix grad == 2W", g, 2 * W)
_check("matrix grad keeps shape", np.array(g.shape), np.array(W.shape))

# 4. f = sum(sin x) -> grad = cos x     (check vs analytic nonlinear)
x = np.array([0.1, 0.7, -1.2, 2.0])
_check("sum(sin x) grad == cos x", numeric_gradient(lambda v: np.sum(np.sin(v)), x), np.cos(x))

[PASS] sum(x^2) grad == 2x
[PASS] sum(c*x) grad == c
[PASS] matrix grad == 2W
[PASS] matrix grad keeps shape
[PASS] sum(sin x) grad == cos x


### Tests — `stable_softmax`

In [39]:
# reuses _check from the numeric_gradient test cell above
x = np.array([2.0, 1.0, 0.1])
p = stable_softmax(x)

_check("probs sum to 1", p.sum(), 1.0)
_check("all in (0, 1)", np.all((p > 0) & (p < 1)), True)

# matches the naive definition on small, safe inputs
naive = np.exp(x) / np.sum(np.exp(x))
_check("matches naive softmax", p, naive)

# stability + shift-invariance: huge logits stay finite and give the same result
big = stable_softmax(x + 1000)
_check("finite on x + 1000", np.all(np.isfinite(big)), True)
_check("shift-invariant (== p)", big, p)

# monotonic: largest logit keeps the largest probability
_check("argmax preserved", np.argmax(p), np.argmax(x))

[PASS] probs sum to 1
[PASS] all in (0, 1)
[PASS] matches naive softmax
[PASS] finite on x + 1000
[PASS] shift-invariant (== p)
[PASS] argmax preserved


## Step 1 — Linear Layer

In [40]:
class Linear:
    """Fully-connected layer:  Y = X @ W + b

    Shapes:
        X : (batch, n_in)     activations from the previous layer
        W : (n_in, n_out)
        b : (n_out,)
        Y : (batch, n_out)
    """

    def __init__(self, n_in: int, n_out: int, seed: int = 0) -> None:
        rng = np.random.default_rng(seed)
        
        # simple init: standard normal weights, zero bias
        self.W: np.ndarray = rng.standard_normal((n_in, n_out))
        self.b: np.ndarray = np.zeros(n_out)
            
        # caches / gradient buffers (filled during forward/backward)
        self.X: np.ndarray | None = None   # previous layer's activations, saved for backward
        self.d_w: np.ndarray | None = None
        self.d_b: np.ndarray | None = None

    def forward(self, X: np.ndarray) -> np.ndarray:
        """X: (batch, n_in) -> Y: (batch, n_out)."""
        self.X = X                     # cache X: backward needs it to compute d_w
        return X @ self.W + self.b     # b (n_out,) broadcasts across every row

    def backward(self, d_y: np.ndarray) -> np.ndarray:
        """d_y: (batch, n_out) upstream gradient ∂L/∂Y. Returns d_x: (batch, n_in)."""
        
        # d_w: chain rule + sum over the batch -> X.T @ d_y.
        #   local derivative ∂Y/∂W is X; a weight is reused across all samples,
        #   so the matmul sums those per-sample contributions. Shape (n_in, n_out).
        self.d_w = self.X.T @ d_y

        # d_b: local derivative ∂Y/∂b is 1, so just sum d_y over the batch axis.
        #   Shape (n_out,) -- one gradient per bias.
        self.d_b = d_y.sum(axis=0)

        # d_x: gradient to hand back to the previous layer (becomes its d_y).
        #   local derivative ∂Y/∂X is W, so d_x = d_y @ W.T. Shape (batch, n_in).
        d_x = d_y @ self.W.T
        
        return d_x

    def parameters(self) -> list[tuple[np.ndarray, np.ndarray]]:
        """(value, gradient) pairs for the optimizer: [(W, d_w), (b, d_b)].

        The optimizer reads this each step: values are mutated in place, gradients
        are re-fetched (backward rebinds d_w/d_b to new arrays every call).
        """
        return [(self.W, self.d_w), (self.b, self.d_b)]

    def zero_grad(self) -> None:
        """Reset gradient buffers (called after each optimizer step)."""
        self.d_w = None
        self.d_b = None

### Test — `Linear` gradient check

In [41]:
# Uses numeric_gradient + _check from Step 0.
# Trick: wrap the layer in a SCALAR loss  L = sum(Y * d_y)  (so ∂L/∂Y = d_y),
# then numeric_gradient of L w.r.t. each of W, b, X must match backward().
def gradient_check_linear(n_in=4, n_out=3, batch=5, seed=1):
    rng = np.random.default_rng(seed)
    layer = Linear(n_in, n_out)
    X  = rng.standard_normal((batch, n_in))
    d_y = rng.standard_normal((batch, n_out))     # random upstream (not all-ones)

    W0, b0 = layer.W.copy(), layer.b.copy()

    # analytic gradients from the layer's own backward
    layer.forward(X)
    d_x = layer.backward(d_y)
    d_w_analytic, d_b_analytic = layer.d_w.copy(), layer.d_b.copy()

    # numeric d_w: vary W, hold X/b fixed
    def loss_W(W_flat):
        layer.W = W_flat.reshape(W0.shape)
        return np.sum(layer.forward(X) * d_y)
    d_w_numeric = numeric_gradient(loss_W, W0.copy()); layer.W = W0.copy()

    # numeric d_b: vary b
    def loss_b(b_flat):
        layer.b = b_flat.reshape(b0.shape)
        return np.sum(layer.forward(X) * d_y)
    d_b_numeric = numeric_gradient(loss_b, b0.copy()); layer.b = b0.copy()

    # numeric d_x: vary X, params at originals
    def loss_X(X_flat):
        return np.sum(layer.forward(X_flat.reshape(X.shape)) * d_y)
    d_x_numeric = numeric_gradient(loss_X, X.copy())

    _check("∂Linear/∂W", d_w_analytic, d_w_numeric)
    _check("∂Linear/∂b", d_b_analytic, d_b_numeric)
    _check("∂Linear/∂X", d_x,   d_x_numeric)

gradient_check_linear()

[PASS] ∂Linear/∂W
[PASS] ∂Linear/∂b
[PASS] ∂Linear/∂X


## Step 2 — Cross-Entropy Loss

In [42]:
def cross_entropy(
    raw_class_scores: np.ndarray,
    correct_class_indices: np.ndarray,
) -> tuple[float, np.ndarray]:
    """Softmax cross-entropy for a batch of classification examples.

    Args:
        raw_class_scores      : (number_of_examples_in_batch, number_of_classes)
                              raw scores (logits), one row per example
        correct_class_indices : (number_of_examples_in_batch,)
                              correct class index for each example

    Returns:
        mean_loss          : float        mean cross-entropy over the batch
        gradient_wrt_scores : np.ndarray   (number_of_examples_in_batch, number_of_classes)
                            ∂L/∂scores = (softmax - onehot) / number_of_examples_in_batch
    """
    number_of_examples_in_batch: int = raw_class_scores.shape[0]
    example_rows: np.ndarray = np.arange(number_of_examples_in_batch)   # [0, 1, ..., batch-1], to index each row

    # --- stable softmax, per row (subtract each row's max -> no overflow) ---
    stabilized_scores: np.ndarray = raw_class_scores - raw_class_scores.max(axis=1, keepdims=True)
    exponentiated_scores: np.ndarray = np.exp(stabilized_scores)
    class_probabilities: np.ndarray = exponentiated_scores / exponentiated_scores.sum(axis=1, keepdims=True)

    # --- loss: -log(prob of the correct class), averaged over the batch ---
    probability_of_correct_class: np.ndarray = class_probabilities[example_rows, correct_class_indices]
    negative_log_prob_of_correct_class: np.ndarray = -np.log(probability_of_correct_class)   # (batch,)
    mean_loss: float = float(negative_log_prob_of_correct_class.mean())

    # --- gradient: the clean combined form  softmax - onehot  ---
    #   start from probabilities, subtract 1 at each row's correct class, average over batch
    gradient_wrt_scores: np.ndarray = class_probabilities.copy()
    gradient_wrt_scores[example_rows, correct_class_indices] -= 1
    gradient_wrt_scores /= number_of_examples_in_batch

    return mean_loss, gradient_wrt_scores

### Test — `cross_entropy` gradient check

In [43]:
# The combined gradient softmax - onehot must match finite differences of the loss.
def gradient_check_cross_entropy(number_of_examples_in_batch=4, number_of_classes=5, seed=1):
    random_generator = np.random.default_rng(seed)
    random_scores = random_generator.standard_normal((number_of_examples_in_batch, number_of_classes))
    correct_class_indices = random_generator.integers(0, number_of_classes, size=number_of_examples_in_batch)

    analytic_loss, analytic_gradient = cross_entropy(random_scores, correct_class_indices)

    # numeric gradient: loss as a scalar function of the scores
    def loss_as_function_of_scores(flattened_scores):
        loss_value, _ = cross_entropy(flattened_scores.reshape(random_scores.shape), correct_class_indices)
        return loss_value
    numeric_gradient_value = numeric_gradient(loss_as_function_of_scores, random_scores.copy())

    _check("∂CE/∂scores", analytic_gradient, numeric_gradient_value)

gradient_check_cross_entropy()

[PASS] ∂CE/∂scores


## Step 2 — Adam Optimizer

In [44]:
class Adam:
    """Adam optimizer: updates every parameter of the given layers in place.

    Holds per-parameter state (first/second moment estimates) and shared state
    (timestep, learning rate, betas, epsilon). Each parameter's update is fully
    independent -- the optimizer just loops over all of them.

    Design:
        - references the layers, so it can reach every parameter
        - reads gradients FRESH each step via layer.parameters()
          (backward rebinds d_w/d_b to new arrays every call)
        - updates values IN PLACE (value -= ...), so the layer's W / b actually change
    """

    def __init__(
        self,
        layers: list,
        lr: float = 1e-3,
        beta1: float = 0.9,
        beta2: float = 0.999,
        eps: float = 1e-8,
    ) -> None:
        self.layers: list = layers
        self.lr: float = lr
        self.beta1: float = beta1
        self.beta2: float = beta2
        self.eps: float = eps
        self.time_step: int = 0

        # one moment buffer per parameter, in the stable order parameters() yields.
        # sized from the VALUES (grads may still be None before the first backward).
        self.first_moment_estimates: list[np.ndarray] = []
        self.second_moment_estimates: list[np.ndarray] = []
        for layer in self.layers:
            for (parameter_value, _parameter_gradient) in layer.parameters():
                self.first_moment_estimates.append(np.zeros_like(parameter_value))
                self.second_moment_estimates.append(np.zeros_like(parameter_value))

    def step(self) -> None:
        """Apply one Adam update to every parameter, using the current gradients."""
        self.time_step += 1
        bias_correction_first: float = 1.0 - self.beta1 ** self.time_step
        bias_correction_second: float = 1.0 - self.beta2 ** self.time_step

        parameter_index: int = 0
        for layer in self.layers:
            for (parameter_value, parameter_gradient) in layer.parameters():   # fresh grads
                first_moment: np.ndarray = self.first_moment_estimates[parameter_index]
                second_moment: np.ndarray = self.second_moment_estimates[parameter_index]

                # update biased moment estimates (in place, so buffer identity is kept)
                first_moment *= self.beta1
                first_moment += (1.0 - self.beta1) * parameter_gradient
                second_moment *= self.beta2
                second_moment += (1.0 - self.beta2) * (parameter_gradient ** 2)

                # bias-corrected estimates (early steps would otherwise be too small)
                corrected_first_moment: np.ndarray = first_moment / bias_correction_first
                corrected_second_moment: np.ndarray = second_moment / bias_correction_second

                # in-place parameter update -> mutates the layer's W / b
                parameter_value -= self.lr * corrected_first_moment / (np.sqrt(corrected_second_moment) + self.eps)

                parameter_index += 1

    def zero_grad(self) -> None:
        """Reset every layer's gradient buffers after a step."""
        for layer in self.layers:
            layer.zero_grad()

### Test — `Adam` reduces loss on a learnable task

In [45]:
# Full training loop on a linearly-separable task: a single Linear + cross_entropy,
# optimized by Adam. Labels come from a linear rule, so the model can actually fit them.
def test_adam_reduces_loss(seed=0):
    random_generator = np.random.default_rng(seed)
    number_of_examples, number_of_features, number_of_classes = 64, 5, 3

    inputs = random_generator.standard_normal((number_of_examples, number_of_features))
    true_weights = random_generator.standard_normal((number_of_features, number_of_classes))
    correct_class_indices = np.argmax(inputs @ true_weights, axis=1)   # learnable labels

    layer = Linear(number_of_features, number_of_classes)
    optimizer = Adam([layer], lr=0.1)

    weights_before = layer.W.copy()
    first_loss = None
    last_loss = None
    for step_index in range(300):
        raw_class_scores = layer.forward(inputs)
        loss, gradient_wrt_scores = cross_entropy(raw_class_scores, correct_class_indices)
        layer.backward(gradient_wrt_scores)
        optimizer.step()
        optimizer.zero_grad()
        if step_index == 0:
            first_loss = loss
        last_loss = loss

    predictions = np.argmax(layer.forward(inputs), axis=1)
    accuracy = float((predictions == correct_class_indices).mean())

    _check("Adam drives loss down", last_loss < first_loss * 0.5, True)
    _check("Adam updated the weights in place", np.any(layer.W != weights_before), True)
    _check("fits the learnable task (acc > 0.9)", accuracy > 0.9, True)
    print(f"   loss: {first_loss:.3f} -> {last_loss:.3f}   accuracy: {accuracy:.2f}")

test_adam_reduces_loss()

[PASS] Adam drives loss down
[PASS] Adam updated the weights in place
[PASS] fits the learnable task (acc > 0.9)
   loss: 2.672 -> 0.065   accuracy: 0.98


## CharTokenizer — text ↔ tokens

Character-level tokenizer: converts text to integer token ids and back. Not a Layer — it has
no parameters and the optimizer never touches it. Vocabulary = special tokens
(`<pad>`, `<bos>`, `<eos>`) + every unique character in the training text.

In [46]:
class CharTokenizer:
    """Character-level tokenizer: text <-> list of integer token ids.

    Vocabulary = special tokens + every unique character in the training text.
    Not a Layer: no parameters, never seen by the optimizer.
    """

    PAD_TOKEN: str = "<pad>"
    BOS_TOKEN: str = "<bos>"   # beginning of sequence
    EOS_TOKEN: str = "<eos>"   # end of sequence

    def __init__(self, training_text: str) -> None:
        special_tokens: list[str] = [self.PAD_TOKEN, self.BOS_TOKEN, self.EOS_TOKEN]
        unique_characters: list[str] = sorted(set(training_text))
        # id -> token (list index is the id); token -> id (inverse map)
        self.id_to_token: list[str] = special_tokens + unique_characters
        self.token_to_id: dict[str, int] = {token: token_id for token_id, token in enumerate(self.id_to_token)}

    @property
    def vocab_size(self) -> int:
        return len(self.id_to_token)

    @property
    def pad_id(self) -> int:
        return self.token_to_id[self.PAD_TOKEN]

    @property
    def bos_id(self) -> int:
        return self.token_to_id[self.BOS_TOKEN]

    @property
    def eos_id(self) -> int:
        return self.token_to_id[self.EOS_TOKEN]

    def encode(self, text: str, add_specials: bool = True) -> list[int]:
        """text -> token ids. If add_specials, wrap as <bos> ... <eos>."""
        token_ids: list[int] = [self.token_to_id[character] for character in text]
        if add_specials:
            token_ids = [self.bos_id] + token_ids + [self.eos_id]
        return token_ids

    def decode(self, token_ids: list[int], skip_specials: bool = True) -> str:
        """token ids -> text. If skip_specials, drop <pad>/<bos>/<eos>."""
        special_token_ids: set[int] = {self.pad_id, self.bos_id, self.eos_id}
        return "".join(
            self.id_to_token[token_id]
            for token_id in token_ids
            if not (skip_specials and token_id in special_token_ids)
        )

### Test — `CharTokenizer` round-trip

In [47]:
# _check uses np.allclose (numeric); add a plain-equality check for strings/ints/lists.
def _check_eq(name, got, want):
    ok = (got == want)
    print(f"[{'PASS' if ok else 'FAIL'}] {name}")
    if not ok:
        print("   got :", repr(got))
        print("   want:", repr(want))

def test_char_tokenizer():
    tokenizer = CharTokenizer("a dog runs")

    # vocab = 3 special tokens + unique characters of the training text
    unique_character_count = len(set("a dog runs"))
    _check_eq("vocab size", tokenizer.vocab_size, 3 + unique_character_count)

    # round-trip: decode(encode(text)) recovers the text (specials skipped on decode)
    for text in ["a dog", "runs", "a", "dog runs"]:
        _check_eq(f"round-trip {text!r}", tokenizer.decode(tokenizer.encode(text)), text)

    # encode wraps with <bos> ... <eos>
    token_ids = tokenizer.encode("a")
    _check_eq("starts with <bos>", token_ids[0], tokenizer.bos_id)
    _check_eq("ends with <eos>", token_ids[-1], tokenizer.eos_id)

    # decode can keep the special tokens when asked
    _check_eq("decode keeps specials", tokenizer.decode(token_ids, skip_specials=False), "<bos>a<eos>")

    # add_specials=False encodes just the characters
    _check_eq("no specials when off", tokenizer.encode("a", add_specials=False), [tokenizer.token_to_id["a"]])

test_char_tokenizer()

[PASS] vocab size
[PASS] round-trip 'a dog'
[PASS] round-trip 'runs'
[PASS] round-trip 'a'
[PASS] round-trip 'dog runs'
[PASS] starts with <bos>
[PASS] ends with <eos>
[PASS] decode keeps specials
[PASS] no specials when off


## Embedding — token id → learned vector

A Layer with **one** parameter: the table `(vocab_size, d)`, one row per token.
- **forward** is a row lookup: `table[ids]` (no computation, just indexing).
- **backward** is a **scatter-add**: each used row receives the gradient of every position
  that used it (duplicate ids accumulate). It returns **`None`** — integer ids aren't
  differentiable, and Embedding is always the first layer.

Same `parameters()` contract as `Linear`, so `Adam` updates its table with no special-casing.

In [48]:
class Embedding:
    """Embedding layer: maps integer token ids to learned vectors (a row lookup).

    One trainable parameter: table (vocab_size, embedding_dim), one row per token.
        forward(ids)  -> table[ids]   (rows selected; caches ids)
        backward(d_out) -> None        (scatter-add into d_table; duplicate ids accumulate)

    Returns no input gradient — integer ids aren't differentiable, and Embedding is the
    first layer. Same parameters() contract as Linear, so the optimizer treats it the same.
    """

    def __init__(self, vocab_size: int, embedding_dim: int, seed: int = 0) -> None:
        random_generator = np.random.default_rng(seed)
        self.table: np.ndarray = random_generator.standard_normal((vocab_size, embedding_dim))
        # cache + gradient buffer (filled during forward/backward)
        self.token_ids: np.ndarray | None = None
        self.d_table: np.ndarray | None = None

    def forward(self, token_ids: np.ndarray) -> np.ndarray:
        """token_ids: integer array of shape S -> embeddings of shape S + (embedding_dim,)."""
        self.token_ids = token_ids          # cache ids: backward needs to know which rows were used
        return self.table[token_ids]       # fancy indexing = row lookup

    def backward(self, d_out: np.ndarray) -> None:
        """d_out: ∂L/∂(looked-up rows), same shape as the forward output.

        Scatter-add each position's gradient into the row of its id. np.add.at accumulates
        on repeated ids (plain d_table[ids] = d_out would overwrite duplicates -> wrong).
        Returns None: there is no gradient to pass back to integer ids.
        """
        self.d_table = np.zeros_like(self.table)
        np.add.at(self.d_table, self.token_ids, d_out)
        return None

    def parameters(self) -> list[tuple[np.ndarray, np.ndarray]]:
        """(value, gradient) pairs for the optimizer: [(table, d_table)]."""
        return [(self.table, self.d_table)]

    def zero_grad(self) -> None:
        """Reset the gradient buffer (called after each optimizer step)."""
        self.d_table = None

### Test — `Embedding` gradient check (scatter-add)

In [49]:
# Scatter-add backward must match finite differences of the table.
# Use a REPEATED id so the accumulation path is exercised.
def gradient_check_embedding(vocab_size=6, embedding_dim=4, seed=1):
    random_generator = np.random.default_rng(seed)
    embedding = Embedding(vocab_size, embedding_dim)

    token_ids = np.array([2, 0, 4, 2, 2])   # id 2 appears 3x -> tests scatter-add accumulation
    upstream_gradient = random_generator.standard_normal((len(token_ids), embedding_dim))

    # forward shape sanity
    embedded_rows = embedding.forward(token_ids)
    _check_eq("forward shape", embedded_rows.shape, (len(token_ids), embedding_dim))

    # analytic gradient from scatter-add backward
    return_value = embedding.backward(upstream_gradient)
    analytic_gradient = embedding.d_table.copy()
    _check_eq("backward returns None", return_value, None)

    # accumulation: the repeated id's row = sum of the gradients at its occurrences (0, 3, 4)
    _check("scatter-add accumulates", analytic_gradient[2], upstream_gradient[[0, 3, 4]].sum(axis=0))

    # numeric gradient: L = sum(table[ids] * upstream_gradient) as a function of the table
    original_table = embedding.table.copy()
    def loss_as_function_of_table(flattened_table):
        embedding.table = flattened_table.reshape(original_table.shape)
        return np.sum(embedding.forward(token_ids) * upstream_gradient)
    numeric_gradient_value = numeric_gradient(loss_as_function_of_table, original_table.copy())
    embedding.table = original_table

    _check("∂Embedding/∂table", analytic_gradient, numeric_gradient_value)

gradient_check_embedding()

[PASS] forward shape
[PASS] backward returns None
[PASS] scatter-add accumulates
[PASS] ∂Embedding/∂table


## Bigram — next token from the current token

The first assembled model, and the integration test for everything so far. It predicts the
next token from **only the current token**: `Embedding(vocab, d) → Linear(d, vocab) → logits`.

- **`forward`** (both phases) — embed the current token, project to next-token logits.
- **`backward`** (training) — push the loss gradient back through Linear, then Embedding.
- **`generate`** (working) — forward only: sample a token, feed it back, until `<eos>`.

`parameters()` / `zero_grad()` just delegate to the two sub-layers, so `Adam([model])` updates
both with no special-casing.

In [50]:
class Bigram:
    """Bigram language model: predicts the next token from ONLY the current token.

    Composition:  Embedding(vocab, d) -> Linear(d, vocab) -> next-token logits.
    forward + backward are used in training; forward alone drives generation.
    """

    def __init__(self, vocab_size: int, embedding_dim: int, seed: int = 0) -> None:
        self.vocab_size: int = vocab_size
        self.embedding: Embedding = Embedding(vocab_size, embedding_dim, seed=seed)
        self.projection: Linear = Linear(embedding_dim, vocab_size, seed=seed + 1)

    def forward(self, current_token_ids: np.ndarray) -> np.ndarray:
        """current_token_ids: (batch,) -> logits (batch, vocab_size)."""
        embedded: np.ndarray = self.embedding.forward(current_token_ids)   # (batch, d)
        logits: np.ndarray = self.projection.forward(embedded)          # (batch, vocab)
        return logits

    def backward(self, gradient_wrt_logits: np.ndarray) -> None:
        """Backprop the loss gradient through projection, then embedding."""
        gradient_wrt_embedded: np.ndarray = self.projection.backward(gradient_wrt_logits)  # (batch, d)
        self.embedding.backward(gradient_wrt_embedded)   # scatter-add into the table; returns None
        return None

    def parameters(self) -> list[tuple[np.ndarray, np.ndarray]]:
        """All (value, gradient) pairs, so one optimizer can update the whole model."""
        return self.embedding.parameters() + self.projection.parameters()

    def zero_grad(self) -> None:
        self.embedding.zero_grad()
        self.projection.zero_grad()

    def generate(
        self,
        tokenizer: CharTokenizer,
        max_new_tokens: int = 100,
        temperature: float = 1.0,
        seed: int = 0,
    ) -> str:
        """Autoregressive generation: start at <bos>, sample tokens until <eos> or the cap."""
        random_generator = np.random.default_rng(seed)
        generated_ids: list[int] = [tokenizer.bos_id]

        for _ in range(max_new_tokens):
            current_token_id: np.ndarray = np.array([generated_ids[-1]])       # (1,)
            logits: np.ndarray = self.forward(current_token_id)[0]           # (vocab,)
            probabilities: np.ndarray = stable_softmax(logits / temperature)
            probabilities = probabilities / probabilities.sum()            # guard against rounding
            next_token_id: int = int(random_generator.choice(self.vocab_size, p=probabilities))
            generated_ids.append(next_token_id)
            if next_token_id == tokenizer.eos_id:
                break

        return tokenizer.decode(generated_ids)

### Train + generate — first text from a from-scratch LM

Train on a small structured corpus with the full stack (`Embedding → Linear → cross_entropy →
Adam`) on shift-by-one pairs, then generate. Success = loss beats the uniform baseline
`ln(vocab)` and the output forms letter patterns.

In [51]:
def train_bigram(
    model: Bigram,
    tokenizer: CharTokenizer,
    text: str,
    steps: int = 500,
    lr: float = 0.1,
) -> list[float]:
    """Full-batch training on shift-by-one pairs. Returns the loss at each step."""
    token_ids: np.ndarray = np.array(tokenizer.encode(text))   # includes <bos> ... <eos>
    current_token_ids: np.ndarray = token_ids[:-1]               # inputs
    next_token_ids: np.ndarray = token_ids[1:]                   # targets (shifted by one)

    optimizer = Adam([model], lr=lr)
    loss_history: list[float] = []
    for _ in range(steps):
        logits = model.forward(current_token_ids)
        loss, gradient_wrt_logits = cross_entropy(logits, next_token_ids)
        model.backward(gradient_wrt_logits)
        optimizer.step()
        optimizer.zero_grad()
        loss_history.append(loss)
    return loss_history


def test_bigram():
    corpus_text = "the quick brown fox jumps over the lazy dog. " * 30
    tokenizer = CharTokenizer(corpus_text)
    model = Bigram(tokenizer.vocab_size, embedding_dim=32)

    loss_history = train_bigram(model, tokenizer, corpus_text, steps=500, lr=0.1)

    uniform_baseline = float(np.log(tokenizer.vocab_size))   # loss of random guessing
    _check("loss decreased", loss_history[-1] < loss_history[0], True)
    _check("beats uniform baseline ln(vocab)", loss_history[-1] < uniform_baseline, True)
    print(f"   loss: {loss_history[0]:.3f} -> {loss_history[-1]:.3f}   (uniform baseline {uniform_baseline:.3f})")

    sample = model.generate(tokenizer, max_new_tokens=60, temperature=0.8, seed=1)
    print("   sample:", repr(sample))

test_bigram()

[PASS] loss decreased
[PASS] beats uniform baseline ln(vocab)
   loss: 12.641 -> 0.639   (uniform baseline 3.434)
   sample: 'ther juick ove lazy quick overog. ox lazy ove lazy the brox '


## CausalSelfAttention — mixing information across positions

The heart of the model. Each position builds a **query**, compares it against every **key**,
and takes a weighted average of the **values**:

$$\text{attention}(Q,K,V) = \text{softmax}\!\left(\frac{QK^\top}{\sqrt{d_{head}}} + \text{mask}\right)V$$

- **Q, K, V** are three `Linear` projections of the same input (reusing our `Linear` layer).
- **`/√d_head`** stops scores from growing with dimension, which would saturate softmax.
- The **causal mask** sets future positions to `−∞` *before* softmax, so position `i` attends
  only to positions `≤ i`. That is what makes the model autoregressive.

Shapes are **single-sequence** `(T, d_model)` — one sequence at a time. That keeps the
backward pass (the hardest part of the project) simple enough to gradient-check directly.

In [52]:
def softmax_rows(scores: np.ndarray) -> np.ndarray:
    """Row-wise stable softmax: every ROW becomes its own distribution summing to 1.

    stable_softmax normalises over the whole array; attention needs one distribution per
    query row, so max/sum are taken along the last axis only. Rows may contain -inf
    (masked positions): exp(-inf) = 0, so those receive exactly zero probability.
    """
    shifted: np.ndarray = scores - scores.max(axis=-1, keepdims=True)
    exponentiated: np.ndarray = np.exp(shifted)
    return exponentiated / exponentiated.sum(axis=-1, keepdims=True)


class CausalSelfAttention:
    """Single-head causal self-attention over ONE sequence.

        forward :  X     (T, d_model)  ->  output (T, d_head)
        backward:  d_out (T, d_head)   ->  d_x    (T, d_model)

    output = softmax(Q @ K.T / sqrt(d_head) + causal_mask) @ V

    Q, K and V are three Linear projections of the SAME X, so the gradient flowing back
    into X is the SUM of those three paths.
    """

    def __init__(self, d_model: int, d_head: int, seed: int = 0) -> None:
        self.d_model: int = d_model
        self.d_head: int = d_head
        # three independent projections of the same input (reuse our Linear layer)
        self.query_projection: Linear = Linear(d_model, d_head, seed=seed)
        self.key_projection: Linear = Linear(d_model, d_head, seed=seed + 1)
        self.value_projection: Linear = Linear(d_model, d_head, seed=seed + 2)
        # caches filled by forward, consumed by backward
        self.Q: np.ndarray | None = None
        self.K: np.ndarray | None = None
        self.V: np.ndarray | None = None
        self.attention_weights: np.ndarray | None = None

    def forward(self, X: np.ndarray) -> np.ndarray:
        """X: (T, d_model) -> output: (T, d_head)."""
        sequence_length: int = X.shape[0]

        self.Q = self.query_projection.forward(X)   # (T, d_head)  "what am I looking for?"
        self.K = self.key_projection.forward(X)     # (T, d_head)  "what do I offer?"
        self.V = self.value_projection.forward(X)   # (T, d_head)  "what do I pass on?"

        # every query against every key; /sqrt(d_head) keeps softmax out of saturation
        raw_scores: np.ndarray = (self.Q @ self.K.T) / np.sqrt(self.d_head)

        # causal mask: position i may only attend to positions j <= i (lower triangle)
        allowed_positions: np.ndarray = np.tril(
            np.ones((sequence_length, sequence_length), dtype=bool)
        )
        masked_scores: np.ndarray = np.where(allowed_positions, raw_scores, -np.inf)

        self.attention_weights = softmax_rows(masked_scores)   # (T, T), rows sum to 1
        return self.attention_weights @ self.V                 # (T, d_head)

    def backward(self, d_out: np.ndarray) -> np.ndarray:
        """d_out: ∂L/∂output (T, d_head). Returns d_x: ∂L/∂X (T, d_model)."""

        # 1. through the weighted sum  output = attention_weights @ V
        d_attention_weights: np.ndarray = d_out @ self.V.T          # (T, T)
        d_v: np.ndarray = self.attention_weights.T @ d_out          # (T, d_head)

        # 2. through the row-wise softmax:  ∂L/∂s = p * (∂L/∂p - Σ(∂L/∂p * p))
        #    masked entries have p = 0, so their gradient stays 0 -> no leak to the future.
        row_weighted_sum: np.ndarray = (
            d_attention_weights * self.attention_weights
        ).sum(axis=-1, keepdims=True)
        d_masked_scores: np.ndarray = self.attention_weights * (
            d_attention_weights - row_weighted_sum
        )

        # 3. through the 1/sqrt(d_head) scaling
        d_raw_scores: np.ndarray = d_masked_scores / np.sqrt(self.d_head)

        # 4. through raw_scores = Q @ K.T
        d_q: np.ndarray = d_raw_scores @ self.K                     # (T, d_head)
        d_k: np.ndarray = d_raw_scores.T @ self.Q                   # (T, d_head)

        # 5. through the three projections; X fed all three, so the paths ADD
        d_x: np.ndarray = (
            self.query_projection.backward(d_q)
            + self.key_projection.backward(d_k)
            + self.value_projection.backward(d_v)
        )
        return d_x

    def parameters(self) -> list[tuple[np.ndarray, np.ndarray]]:
        return (
            self.query_projection.parameters()
            + self.key_projection.parameters()
            + self.value_projection.parameters()
        )

    def zero_grad(self) -> None:
        self.query_projection.zero_grad()
        self.key_projection.zero_grad()
        self.value_projection.zero_grad()

### Test — `CausalSelfAttention` gradient check + causality

Two independent things must hold:
1. **Gradients** — the hand-derived backward (softmax Jacobian, `Q Kᵀ`, the three
   projections) matches finite differences for `W_q`, `W_k`, `W_v` and `X`.
2. **Causality** — changing a *later* token must not change *earlier* outputs, attention rows
   must sum to 1, and the strict upper triangle must be exactly zero.

In [53]:
# Scalar-loss trick again: L = sum(output * upstream_gradient), so ∂L/∂output = upstream.
def gradient_check_attention(sequence_length=5, d_model=6, d_head=4, seed=1):
    random_generator = np.random.default_rng(seed)
    attention = CausalSelfAttention(d_model, d_head)
    X = random_generator.standard_normal((sequence_length, d_model))
    upstream_gradient = random_generator.standard_normal((sequence_length, d_head))

    output = attention.forward(X)
    _check_eq("attention output shape", output.shape, (sequence_length, d_head))

    d_x_analytic = attention.backward(upstream_gradient)
    analytic_by_name = {
        "∂attn/∂W_q": (attention.query_projection, attention.query_projection.d_w.copy()),
        "∂attn/∂W_k": (attention.key_projection, attention.key_projection.d_w.copy()),
        "∂attn/∂W_v": (attention.value_projection, attention.value_projection.d_w.copy()),
    }

    # numeric gradient for each projection weight, one at a time (others held fixed)
    for name, (projection, analytic_gradient) in analytic_by_name.items():
        original_weight = projection.W.copy()

        def loss_as_function_of_weight(flattened_weight):
            projection.W = flattened_weight.reshape(original_weight.shape)
            return np.sum(attention.forward(X) * upstream_gradient)

        numeric_gradient_value = numeric_gradient(loss_as_function_of_weight, original_weight.copy())
        projection.W = original_weight          # restore before the next check
        _check(name, analytic_gradient, numeric_gradient_value)

    # numeric gradient w.r.t. the input X (all three paths must sum correctly)
    def loss_as_function_of_input(flattened_input):
        return np.sum(attention.forward(flattened_input.reshape(X.shape)) * upstream_gradient)

    _check("∂attn/∂X", d_x_analytic, numeric_gradient(loss_as_function_of_input, X.copy()))


def test_attention_causality(sequence_length=6, d_model=8, d_head=4, seed=2):
    random_generator = np.random.default_rng(seed)
    attention = CausalSelfAttention(d_model, d_head)
    X = random_generator.standard_normal((sequence_length, d_model))

    output_before = attention.forward(X).copy()
    attention_weights = attention.attention_weights.copy()

    # rows are distributions, and nothing attends to the future
    _check("attention rows sum to 1", attention_weights.sum(axis=1), np.ones(sequence_length))
    upper_triangle = attention_weights[np.triu_indices(sequence_length, k=1)]
    _check("no attention to the future", upper_triangle, np.zeros(upper_triangle.size))

    # perturb ONLY the last token: earlier outputs must be untouched
    perturbed_input = X.copy()
    perturbed_input[-1] += 10.0
    output_after = attention.forward(perturbed_input).copy()

    _check("earlier outputs unchanged by a future token", output_before[:-1], output_after[:-1])
    _check_eq(
        "last output did change",
        bool(np.any(np.abs(output_before[-1] - output_after[-1]) > 1e-8)),
        True,
    )


gradient_check_attention()
test_attention_causality()

[PASS] attention output shape
[PASS] ∂attn/∂W_q
[PASS] ∂attn/∂W_k
[PASS] ∂attn/∂W_v
[PASS] ∂attn/∂X
[PASS] attention rows sum to 1
[PASS] no attention to the future
[PASS] earlier outputs unchanged by a future token
[PASS] last output did change


## MultiHeadAttention — several attention patterns at once

One head can only learn one way of relating positions. Multi-head runs `h` independent heads
in parallel, each with `d_head = d_model / h`, concatenates their outputs, and mixes them with
one final `Linear`:

```
X ──┬─► head_1 ─┐
    ├─► head_2 ─┤ concat (T, h*d_head) ─► output_projection ─► (T, d_model)
    └─► head_h ─┘
```

Because every head reads the same `X`, the backward pass **sums** the `d_x` from all heads —
the same accumulation rule as Q/K/V inside a single head. Heads are kept as a plain list
(rather than one batched tensor): slower, but the shapes stay obvious and checkable.

In [54]:
class MultiHeadAttention:
    """Several CausalSelfAttention heads in parallel, concatenated and projected.

        forward :  X     (T, d_model) -> output (T, d_model)
        backward:  d_out (T, d_model) -> d_x    (T, d_model)

    Every head sees the same X, so backward SUMS each head's d_x.
    """

    def __init__(self, d_model: int, number_of_heads: int, seed: int = 0) -> None:
        if d_model % number_of_heads != 0:
            raise ValueError(
                f"d_model ({d_model}) must be divisible by number_of_heads ({number_of_heads})"
            )
        self.d_model: int = d_model
        self.number_of_heads: int = number_of_heads
        self.d_head: int = d_model // number_of_heads

        # distinct seeds so heads start differently and can specialise
        self.heads: list[CausalSelfAttention] = [
            CausalSelfAttention(d_model, self.d_head, seed=seed + 10 * head_index)
            for head_index in range(number_of_heads)
        ]
        # mixes the concatenated head outputs back into model width
        self.output_projection: Linear = Linear(d_model, d_model, seed=seed + 7)

    def forward(self, X: np.ndarray) -> np.ndarray:
        """X: (T, d_model) -> output: (T, d_model)."""
        head_outputs: list[np.ndarray] = [head.forward(X) for head in self.heads]
        concatenated_heads: np.ndarray = np.concatenate(head_outputs, axis=-1)  # (T, d_model)
        return self.output_projection.forward(concatenated_heads)

    def backward(self, d_out: np.ndarray) -> np.ndarray:
        """d_out: ∂L/∂output (T, d_model). Returns d_x: ∂L/∂X (T, d_model)."""
        # back through the mixing projection, then split the gradient per head
        d_concatenated: np.ndarray = self.output_projection.backward(d_out)
        per_head_gradients: list[np.ndarray] = np.split(
            d_concatenated, self.number_of_heads, axis=-1
        )

        # every head read the same X -> accumulate their input gradients
        d_x: np.ndarray = np.zeros_like(d_concatenated)
        for head, head_gradient in zip(self.heads, per_head_gradients):
            d_x += head.backward(head_gradient)
        return d_x

    def parameters(self) -> list[tuple[np.ndarray, np.ndarray]]:
        collected: list[tuple[np.ndarray, np.ndarray]] = []
        for head in self.heads:
            collected += head.parameters()
        return collected + self.output_projection.parameters()

    def zero_grad(self) -> None:
        for head in self.heads:
            head.zero_grad()
        self.output_projection.zero_grad()

### Test — `MultiHeadAttention` gradient check + causality

In [55]:
def gradient_check_multi_head(sequence_length=5, d_model=8, number_of_heads=4, seed=3):
    random_generator = np.random.default_rng(seed)
    attention = MultiHeadAttention(d_model, number_of_heads)
    X = random_generator.standard_normal((sequence_length, d_model))
    upstream_gradient = random_generator.standard_normal((sequence_length, d_model))

    output = attention.forward(X)
    _check_eq("multi-head output shape", output.shape, (sequence_length, d_model))
    _check_eq("head count", len(attention.heads), number_of_heads)
    _check_eq("d_head split", attention.d_head, d_model // number_of_heads)

    d_x_analytic = attention.backward(upstream_gradient)
    d_w_out_analytic = attention.output_projection.d_w.copy()
    # W_q of the LAST head: proves the per-head gradient split lines up with the right head
    last_head_projection = attention.heads[-1].query_projection
    d_w_q_last_head_analytic = last_head_projection.d_w.copy()

    # numeric: output projection weight
    original_output_weight = attention.output_projection.W.copy()

    def loss_as_function_of_output_weight(flattened_weight):
        attention.output_projection.W = flattened_weight.reshape(original_output_weight.shape)
        return np.sum(attention.forward(X) * upstream_gradient)

    _check(
        "∂multihead/∂W_out",
        d_w_out_analytic,
        numeric_gradient(loss_as_function_of_output_weight, original_output_weight.copy()),
    )
    attention.output_projection.W = original_output_weight

    # numeric: W_q inside the last head
    original_head_weight = last_head_projection.W.copy()

    def loss_as_function_of_head_weight(flattened_weight):
        last_head_projection.W = flattened_weight.reshape(original_head_weight.shape)
        return np.sum(attention.forward(X) * upstream_gradient)

    _check(
        "∂multihead/∂W_q (last head)",
        d_w_q_last_head_analytic,
        numeric_gradient(loss_as_function_of_head_weight, original_head_weight.copy()),
    )
    last_head_projection.W = original_head_weight

    # numeric: the input (must accumulate across ALL heads)
    def loss_as_function_of_input(flattened_input):
        return np.sum(attention.forward(flattened_input.reshape(X.shape)) * upstream_gradient)

    _check("∂multihead/∂X", d_x_analytic, numeric_gradient(loss_as_function_of_input, X.copy()))

    # causality survives the concat + output projection
    output_before = attention.forward(X).copy()
    perturbed_input = X.copy()
    perturbed_input[-1] += 10.0
    output_after = attention.forward(perturbed_input).copy()
    _check("multi-head stays causal", output_before[:-1], output_after[:-1])

    # rejects an indivisible configuration instead of silently misbehaving
    try:
        MultiHeadAttention(d_model=7, number_of_heads=2)
        _check_eq("rejects indivisible d_model", "no error raised", "ValueError")
    except ValueError:
        _check_eq("rejects indivisible d_model", True, True)


gradient_check_multi_head()

[PASS] multi-head output shape
[PASS] head count
[PASS] d_head split
[PASS] ∂multihead/∂W_out
[PASS] ∂multihead/∂W_q (last head)
[PASS] ∂multihead/∂X
[PASS] multi-head stays causal
[PASS] rejects indivisible d_model


## LayerNorm — keeping activations at a stable scale

Stacking blocks makes activation scales drift: every sub-layer adds into the residual stream, so
values grow with depth and training destabilizes. LayerNorm rescales **each token
independently**, across its feature dimension:

$$\hat{x} = \frac{x - \mu}{\sqrt{\sigma^2 + \varepsilon}}, \qquad y = \gamma\,\hat{x} + \beta$$

- **`μ`, `σ²`** — mean and variance over the **features of one token**, not over the batch as
  BatchNorm does. Each token is normalized on its own, so batch size and sequence length never
  change the result — which is exactly what variable-length sequences need.
- **`γ` (scale), `β` (shift)** — learnable, one per feature. Normalizing throws scale and offset
  away; these let the network put back whatever it actually needs.
- **`ε`** — keeps the division safe when a token is constant (zero variance).

**Why the backward pass is the tricky one.** In `Linear`, each output depends only on its own
inputs. Here `μ` and `σ²` are computed *from every feature*, so nudging one feature moves the
mean and the variance, which shifts **every** output of that token. The gradient therefore has
three terms — the direct path, a mean correction, and a variance correction:

$$\frac{\partial L}{\partial x} = \frac{1}{\sigma}\left[\frac{\partial L}{\partial \hat{x}} - \overline{\left(\frac{\partial L}{\partial \hat{x}}\right)} - \hat{x}\;\overline{\left(\frac{\partial L}{\partial \hat{x}}\hat{x}\right)}\right]$$

(bars are means over the feature axis). This is a classic source of silent bugs, so the gradient
check matters more here than anywhere except attention.

In [ ]:
class LayerNorm:
    """Normalizes each token across its FEATURE dimension, then scales and shifts.

        forward :  X     (..., d_model) -> Y   (..., d_model)
        backward:  d_y   (..., d_model) -> d_x (..., d_model)

    y = gamma * (x - mean) / sqrt(variance + eps) + beta,  with mean/variance taken
    over the last axis (one token), NOT over the batch.

    Two trainable parameters, both of shape (d_model,):
        gamma - per-feature scale (init 1 -> starts as a pure normalizer)
        beta  - per-feature shift (init 0)
    """

    def __init__(self, d_model: int, eps: float = 1e-5) -> None:
        self.d_model: int = d_model
        self.eps: float = eps
        # identity transform at init: scale 1, shift 0
        self.gamma: np.ndarray = np.ones(d_model)
        self.beta: np.ndarray = np.zeros(d_model)
        # caches + gradient buffers
        self.normalized_input: np.ndarray | None = None      # x_hat, needed by backward
        self.standard_deviation: np.ndarray | None = None    # sqrt(variance + eps)
        self.d_gamma: np.ndarray | None = None
        self.d_beta: np.ndarray | None = None

    def forward(self, X: np.ndarray) -> np.ndarray:
        """X: (..., d_model) -> Y: (..., d_model). Normalizes over the LAST axis."""
        feature_mean: np.ndarray = X.mean(axis=-1, keepdims=True)
        feature_variance: np.ndarray = X.var(axis=-1, keepdims=True)   # biased (1/d), as in the paper

        self.standard_deviation = np.sqrt(feature_variance + self.eps)
        self.normalized_input = (X - feature_mean) / self.standard_deviation

        return self.gamma * self.normalized_input + self.beta

    def backward(self, d_y: np.ndarray) -> np.ndarray:
        """d_y: ∂L/∂Y (..., d_model). Returns d_x: ∂L/∂X (..., d_model)."""
        # every axis except the feature axis is summed over: gamma/beta are shared
        # across all tokens, so each receives every token's contribution.
        summed_axes: tuple[int, ...] = tuple(range(d_y.ndim - 1))
        self.d_gamma = (d_y * self.normalized_input).sum(axis=summed_axes)
        self.d_beta = d_y.sum(axis=summed_axes)

        # gradient w.r.t. the normalized values
        d_normalized: np.ndarray = d_y * self.gamma

        # Three terms: the direct path, minus the mean correction, minus the variance
        # correction. They appear because mu and sigma depend on EVERY feature of the
        # token, so perturbing one feature moves all outputs of that token.
        mean_of_d_normalized: np.ndarray = d_normalized.mean(axis=-1, keepdims=True)
        mean_of_d_normalized_times_x_hat: np.ndarray = (
            d_normalized * self.normalized_input
        ).mean(axis=-1, keepdims=True)

        d_x: np.ndarray = (
            d_normalized
            - mean_of_d_normalized
            - self.normalized_input * mean_of_d_normalized_times_x_hat
        ) / self.standard_deviation
        return d_x

    def parameters(self) -> list[tuple[np.ndarray, np.ndarray]]:
        """(value, gradient) pairs for the optimizer: [(gamma, d_gamma), (beta, d_beta)]."""
        return [(self.gamma, self.d_gamma), (self.beta, self.d_beta)]

    def zero_grad(self) -> None:
        self.d_gamma = None
        self.d_beta = None

### Test — `LayerNorm` gradient check + normalization properties

Beyond the gradients, three behavioural properties must hold: rows come out with zero mean and
unit variance, `gamma`/`beta` are applied per feature, and the output is **invariant to shifting
or scaling a whole token** (that is what "normalization" means).

In [ ]:
def test_layer_norm_properties(sequence_length=4, d_model=6, seed=1):
    random_generator = np.random.default_rng(seed)
    layer_norm = LayerNorm(d_model)
    # inputs deliberately off-centre and badly scaled, to prove normalization works
    X = random_generator.standard_normal((sequence_length, d_model)) * 7.0 + 3.0

    output = layer_norm.forward(X)
    _check_eq("layernorm output shape", output.shape, (sequence_length, d_model))

    # with gamma=1, beta=0 the output is exactly the normalized input
    _check("rows have zero mean", output.mean(axis=-1), np.zeros(sequence_length))
    _check("rows have unit variance", output.var(axis=-1), np.ones(sequence_length), atol=1e-4)

    # normalization removes any per-token shift or scale
    shifted_output = layer_norm.forward(X + 100.0)
    _check("invariant to shifting a token", shifted_output, output, atol=1e-6)
    scaled_output = layer_norm.forward(X * 5.0)
    _check("invariant to scaling a token", scaled_output, output, atol=1e-6)

    # gamma / beta are applied per feature
    layer_norm.gamma = random_generator.standard_normal(d_model)
    layer_norm.beta = random_generator.standard_normal(d_model)
    affine_output = layer_norm.forward(X)
    _check(
        "gamma/beta applied per feature",
        affine_output,
        layer_norm.gamma * layer_norm.normalized_input + layer_norm.beta,
    )


def gradient_check_layer_norm(sequence_length=4, d_model=5, seed=2):
    random_generator = np.random.default_rng(seed)
    layer_norm = LayerNorm(d_model)
    # non-trivial gamma/beta so their gradients are actually exercised
    layer_norm.gamma = random_generator.standard_normal(d_model)
    layer_norm.beta = random_generator.standard_normal(d_model)

    X = random_generator.standard_normal((sequence_length, d_model)) * 2.0 + 1.0
    upstream_gradient = random_generator.standard_normal((sequence_length, d_model))

    layer_norm.forward(X)
    d_x_analytic = layer_norm.backward(upstream_gradient)
    d_gamma_analytic = layer_norm.d_gamma.copy()
    d_beta_analytic = layer_norm.d_beta.copy()

    original_gamma = layer_norm.gamma.copy()
    original_beta = layer_norm.beta.copy()

    def loss_as_function_of_gamma(flattened_gamma):
        layer_norm.gamma = flattened_gamma.reshape(original_gamma.shape)
        return np.sum(layer_norm.forward(X) * upstream_gradient)

    _check("∂LayerNorm/∂gamma", d_gamma_analytic,
           numeric_gradient(loss_as_function_of_gamma, original_gamma.copy()))
    layer_norm.gamma = original_gamma

    def loss_as_function_of_beta(flattened_beta):
        layer_norm.beta = flattened_beta.reshape(original_beta.shape)
        return np.sum(layer_norm.forward(X) * upstream_gradient)

    _check("∂LayerNorm/∂beta", d_beta_analytic,
           numeric_gradient(loss_as_function_of_beta, original_beta.copy()))
    layer_norm.beta = original_beta

    # the hard one: the three-term input gradient
    def loss_as_function_of_input(flattened_input):
        return np.sum(layer_norm.forward(flattened_input.reshape(X.shape)) * upstream_gradient)

    _check("∂LayerNorm/∂X", d_x_analytic,
           numeric_gradient(loss_as_function_of_input, X.copy()))


test_layer_norm_properties()
gradient_check_layer_norm()